# EDA d'inspection — la-centrale-fr

Dataset candidat pour le pilier Prix. Cible = `price`.

> ⚠️ **Gate EDA** (cf. `ml/AGENTS.md`) : ce notebook **inspecte** seulement.
> Aucun nettoyage, aucune modélisation. Le fichier `raw/` n'est jamais modifié.
> À la fin : résumé chiffré → **STOP** → validation avant toute suite.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../../data/la-centrale-fr/raw/la_centrale.csv")
df = pd.read_csv(RAW)
print(f"{df.shape[0]} lignes x {df.shape[1]} colonnes")

141199 lignes x 30 colonnes


## 1. Colonnes & types

In [2]:
print("Colonnes:")
for col in df.columns:
    print(f"  - {col}")
print()
df.dtypes

Colonnes:
  - Unnamed: 0
  - model1
  - model2
  - version
  - price
  - km
  - fuel
  - days_after_pub
  - CV_fisc
  - HorseP
  - dep
  - warranty_month
  - CT
  - Crit_Air
  - Gearbox_auto
  - L_by_100km
  - numbe_seats
  - doors_nb
  - Euro_stand
  - first_hand
  - CO2_g_km
  - trunk_volume
  - year
  - circuilation_days
  - Length
  - Nb_option
  - circuilation_date
  - month
  - ID_index
  - dep_name



Unnamed: 0             int64
model1                   str
model2                   str
version                  str
price                  int64
km                     int64
fuel                     str
days_after_pub         int64
CV_fisc              float64
HorseP               float64
dep                    int64
warranty_month         int64
CT                     int64
Crit_Air             float64
Gearbox_auto           int64
L_by_100km           float64
numbe_seats          float64
doors_nb             float64
Euro_stand           float64
first_hand             int64
CO2_g_km             float64
trunk_volume         float64
year                   int64
circuilation_days      int64
Length               float64
Nb_option            float64
circuilation_date        str
month                  int64
ID_index               int64
dep_name                 str
dtype: object

In [3]:
df.head(3)

,Unnamed: 0,model1,model2,version,price,km,fuel,days_after_pub,CV_fisc,HorseP,...,CO2_g_km,trunk_volume,year,circuilation_days,Length,Nb_option,circuilation_date,month,ID_index,dep_name
0,0,HYUNDAI,I20 (3E GENERATION),III 1.0 T-GDI 100 ACTIVE,16340,47981,Essence,60,5.0,100.0,...,NaN,326.0,2018,864,4.07,15.0,2021-07-08 00:00:00.000000,7,0,17 Charente-Maritime
1,1,RENAULT,MEGANE 4,IV 1.2 TCE 130 ENERGY INTENS,16110,104063,Essence,23,7.0,132.0,...,119.0,384.0,2016,2326,4.36,44.0,2017-07-07 00:00:00.000000,7,1,75 Paris
2,2,PEUGEOT,2008,(2) 1.6 BLUEHDI 100 S&S ALLURE BUSINESS,17720,108405,Diesel,27,5.0,99.0,...,90.0,410.0,2016,1635,4.16,33.0,2019-05-29 00:00:00.000000,5,2,76 Seine-Maritime


## 2. Valeurs manquantes (% par colonne)

In [4]:
na_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
na_pct[na_pct > 0]

L_by_100km      14.3
trunk_volume    10.3
numbe_seats      4.3
Length           4.2
Crit_Air         4.2
Euro_stand       4.1
CO2_g_km         3.9
Nb_option        0.6
HorseP           0.3
CV_fisc          0.3
doors_nb         0.1
dtype: float64

## 3. Qualité de la cible `price`

On vérifie le type, les valeurs non positives et la distribution — sans rien filtrer.

In [5]:
price = df["price"]
print("dtype       :", price.dtype)
print("Non positifs :", int((price <= 0).sum()))
print("Manquants    :", int(price.isna().sum()))
print()
print("Describe :")
print(price.describe().round(0))
print()
print("Quantiles hauts :")
print(price.quantile([0.90, 0.95, 0.99]).round(0))

dtype       : int64
Non positifs : 0
Manquants    : 0

Describe :
count    141199.0
mean      35154.0
std       38809.0
min        1740.0
25%       18960.0
50%       26480.0
75%       37000.0
max      793620.0
Name: price, dtype: float64

Quantiles hauts :
0.90     54270.0
0.95     82390.0
0.99    206360.0
Name: price, dtype: float64


## 4. Aberrations & doublons

In [6]:
print("Lignes dupliquees :", df.duplicated().sum())
print()
print("Unnamed: 0 unique :", df["Unnamed: 0"].is_unique)
print("Unnamed: 0 == index par defaut :", bool((df["Unnamed: 0"] == np.arange(len(df))).all()))
print("ID_index unique   :", df["ID_index"].is_unique)
print()
yr = df["year"]
print("year min/max            :", int(yr.min()), int(yr.max()))
print("year hors [1980, 2026]  :", int(((yr < 1980) | (yr > 2026)).sum()))
print()
km = df["km"]
print("km min/max :", int(km.min()), int(km.max()))
print("km == 0    :", int((km == 0).sum()))

Lignes dupliquees : 0

Unnamed: 0 unique : True
Unnamed: 0 == index par defaut : True
ID_index unique   : True

year min/max            : 1953 2022
year hors [1980, 2026]  : 90

km min/max : 1 580859
km == 0    : 0


## 5. Cardinalité des catégorielles clés

In [7]:
for col in ["model1", "model2", "version", "fuel", "Gearbox_auto", "dep_name", "first_hand"]:
    if col in df.columns:
        print(f"--- {col} : {df[col].nunique()} valeurs distinctes ---")
        print(df[col].value_counts(dropna=False).head(5))
        print()

--- model1 : 65 valeurs distinctes ---
model1
PEUGEOT       30285
CITROEN       21417
RENAULT       10757
AUDI           9966
VOLKSWAGEN     8202
Name: count, dtype: int64

--- model2 : 1172 valeurs distinctes ---
model2
C3 (3E GENERATION)      6510
3008 (2E GENERATION)    5779
208 (2E GENERATION)     5175
308 (2E GENERATION)     3170
C3 AIRCROSS             3036
Name: count, dtype: int64

--- version : 8504 valeurs distinctes ---
version
II 1.2 PURETECH 75 S&S ACTIVE BUSINESS    1123
1.5 BLUEHDI 130 S&S EAT8 SHINE PACK        698
II 1.2 PURETECH 100 S&S ALLURE             536
III 1.2 PURETECH 82 S&S SHINE              514
III (2) 1.2 PURETECH 83 S&S SHINE          507
Name: count, dtype: int64

--- fuel : 8 valeurs distinctes ---
fuel
Diesel                        69176
Essence                       59734
Hybride essence électrique     8666
Electrique                     2625
Hybride diesel électrique       393
Name: count, dtype: int64

--- Gearbox_auto : 2 valeurs distinctes ---
Gea

## 6. Résumé chiffré (→ STOP validation)

In [8]:
na_global = df.isna().mean().mean() * 100
tres_vides = int(((df.isna().mean() * 100) > 90).sum())

print("=" * 52)
print("RESUME — la-centrale-fr")
print("=" * 52)
print(f"Dimensions          : {df.shape[0]} lignes x {df.shape[1]} colonnes")
print(f"NA global moyen     : {na_global:.1f} %")
print(f"Colonnes >90% vide  : {tres_vides}")
print(f"Cible price         : {price.dtype}, {int((df['price']<=0).sum())} non positifs, {int(df['price'].isna().sum())} manquants")
print(f"price median / max  : {df['price'].median():.0f} / {df['price'].max():.0f} EUR")
print(f"Lignes dupliquees   : {df.duplicated().sum()}")
print("=" * 52)
print("STOP — attente validation avant nettoyage / regression.")

RESUME — la-centrale-fr
Dimensions          : 141199 lignes x 30 colonnes
NA global moyen     : 1.6 %
Colonnes >90% vide  : 0
Cible price         : int64, 0 non positifs, 0 manquants
price median / max  : 26480 / 793620 EUR


Lignes dupliquees   : 0
STOP — attente validation avant nettoyage / regression.
